# Exploring the Relationship between Health and Educational Outcomes
**Data Analysis with Python — Group Project**  
Cosimo ZATTI, William Manca di Villahermosa, Robert BIRKMAYER

This notebook investigates the association between three OECD health indicators (life expectancy at birth, infant mortality, health expenditure as % of GDP) and country-level PISA 2022 scores in mathematics, reading, and science.

**Structure:**
1. Setup & Data Loading
2. Exploratory Data Analysis
3. Bivariate Analysis — Facet Grid
4. Multivariate OLS Regression
5. Robustness Checks

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- File paths (relative to notebook) ---
PISA_FILE        = Path('xluqor.xlsx')
LIFE_EXP_FILE    = Path('OECD.ELS.HD,DSD_HEALTH_STAT@DF_LE,1.1,filtered,2026-04-23 15-20-54.xlsx')
INFANT_MORT_FILE = Path('OECD.ELS.HD,DSD_HEALTH_STAT@DF_MIM,1.1,filtered,2026-04-23 15-22-00.xlsx')
HEALTH_EXP_FILE  = Path('OECD.ELS.HD,DSD_SHA@DF_SHA,1.0,filtered,2026-04-23 15-25-18.xlsx')

In [ ]:
def load_pisa_subject(filepath, sheet_name, score_col):
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=10, engine='calamine')
    df = raw.iloc[1:, :2].copy()
    df.columns = [score_col, 'country']
    df['country'] = df['country'].astype(str).str.replace('*', '', regex=False).str.strip()
    # Harmonise names that differ between PISA and OECD health data
    df['country'] = df['country'].replace({'Czech Republic': 'Czechia'})
    df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
    return df[['country', score_col]].dropna()


def load_oecd_health(filepath, col_name, year='2022'):
    raw = pd.read_excel(filepath, header=5, engine='calamine')
    df = raw[['Time period', year]].copy()
    df.columns = ['country', col_name]
    skip_exact = {'Reference area', 'Non-OECD economies', '© Terms & conditions'}
    df = df[~df['country'].astype(str).str.contains(':', na=False)]
    df = df[~df['country'].astype(str).str.strip().isin(skip_exact)]
    df['country'] = df['country'].astype(str).str.replace('·', '', regex=False).str.strip()
    df[col_name]  = pd.to_numeric(df[col_name], errors='coerce')
    df = df.dropna().drop_duplicates(subset='country', keep='first')
    return df.reset_index(drop=True)


# Load and merge PISA subjects
pisa = (
    load_pisa_subject(PISA_FILE, 'Table I.2.1', 'math_score')
    .merge(load_pisa_subject(PISA_FILE, 'Table I.2.2', 'reading_score'), on='country')
    .merge(load_pisa_subject(PISA_FILE, 'Table I.2.3', 'science_score'), on='country')
)

# Load and merge health indicators
health = (
    load_oecd_health(LIFE_EXP_FILE,    'life_expectancy')
    .merge(load_oecd_health(INFANT_MORT_FILE, 'infant_mortality'),    on='country', how='outer')
    .merge(load_oecd_health(HEALTH_EXP_FILE,  'health_exp_pct_gdp'), on='country', how='outer')
)

# Final dataset: inner join, drop rows with any missing value
df = pisa.merge(health, on='country', how='inner').dropna().reset_index(drop=True)

print(f"PISA countries:         {len(pisa)}")
print(f"OECD health countries:  {len(health)}")
print(f"Final dataset:          {len(df)} countries (complete data on all 6 variables)")
df

## 2. Exploratory Data Analysis

In [ ]:
# --- 2a. Descriptive statistics ---
df.describe().round(2)

In [ ]:
# --- 2b. Distributions ---
vars_info = {
    'math_score':         'Math Score (PISA points)',
    'reading_score':      'Reading Score (PISA points)',
    'science_score':      'Science Score (PISA points)',
    'life_expectancy':    'Life Expectancy (years)',
    'infant_mortality':   'Infant Mortality (per 1,000)',
    'health_exp_pct_gdp': 'Health Expenditure (% GDP)',
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (col, label) in zip(axes.flat, vars_info.items()):
    ax.hist(df[col], bins=15, edgecolor='white', color='steelblue', alpha=0.85)
    ax.axvline(df[col].mean(),   color='firebrick', linewidth=1.5, linestyle='--', label='Mean')
    ax.axvline(df[col].median(), color='darkorange', linewidth=1.5, linestyle=':',  label='Median')
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('Variable Distributions (n = 43 countries)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 2c. Pearson + Spearman correlations with p-values ---
health_vars = ['life_expectancy', 'infant_mortality', 'health_exp_pct_gdp']
pisa_vars   = ['math_score', 'reading_score', 'science_score']

rows = []
for hv in health_vars:
    for pv in pisa_vars:
        pr, pp = stats.pearsonr(df[hv],  df[pv])
        sr, sp = stats.spearmanr(df[hv], df[pv])
        rows.append({
            'Health indicator': hv.replace('_', ' ').title(),
            'PISA subject':     pv.replace('_score', '').title(),
            'Pearson r':        round(pr, 3),
            'Pearson p':        round(pp, 3),
            'Spearman ρ':       round(sr, 3),
            'Spearman p':       round(sp, 3),
        })

corr_table = pd.DataFrame(rows)
print("Correlations between health indicators and PISA scores (n=43)")
print("* p < 0.05   ** p < 0.01   *** p < 0.001")
print()

def star(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''

corr_table['Pearson r']  = corr_table.apply(lambda r: f"{r['Pearson r']}{star(r['Pearson p'])}",  axis=1)
corr_table['Spearman ρ'] = corr_table.apply(lambda r: f"{r['Spearman ρ']}{star(r['Spearman p'])}", axis=1)
corr_table[['Health indicator', 'PISA subject', 'Pearson r', 'Pearson p', 'Spearman ρ', 'Spearman p']]

In [ ]:
# --- 2d. Full correlation heatmap (Pearson) ---
all_vars = pisa_vars + health_vars
labels   = ['Math', 'Reading', 'Science', 'Life Exp.', 'Infant Mort.', 'Health Exp.']

corr_matrix = df[all_vars].corr()
corr_matrix.index   = labels
corr_matrix.columns = labels

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True   # upper triangle
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, linewidths=0.5, square=True, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Pearson Correlation Matrix', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Bivariate Analysis — Facet Grid

Each panel shows the scatter plot of one health indicator (rows) against one PISA subject (columns), with an OLS regression line, 95% CI, R², and p-value annotated.

In [ ]:
health_labels = {
    'life_expectancy':    'Life Expectancy at Birth (years)',
    'infant_mortality':   'Infant Mortality (per 1,000 live births)',
    'health_exp_pct_gdp': 'Health Expenditure (% of GDP)',
}
pisa_labels = {
    'math_score':    'Mathematics',
    'reading_score': 'Reading',
    'science_score': 'Science',
}
highlight = ['United States', 'Japan', 'Korea', 'Finland', 'Mexico', 'Singapore']

fig, axes = plt.subplots(3, 3, figsize=(14, 12), sharex='col')

for row_idx, (hv, hl) in enumerate(health_labels.items()):
    for col_idx, (pv, pl) in enumerate(pisa_labels.items()):
        ax = axes[row_idx, col_idx]

        sns.regplot(data=df, x=pv, y=hv, ax=ax,
                    scatter_kws=dict(s=40, alpha=0.75, edgecolors='white', linewidths=0.5),
                    line_kws=dict(color='firebrick', linewidth=1.5), ci=95)

        # Annotate notable countries
        for _, row in df[df['country'].isin(highlight)].iterrows():
            ax.annotate(row['country'], xy=(row[pv], row[hv]),
                        xytext=(4, 2), textcoords='offset points',
                        fontsize=7, color='#333333')

        # R² and p-value annotation
        r, p = stats.pearsonr(df[pv], df[hv])
        r2 = r**2
        p_str = f'p={p:.3f}' if p >= 0.001 else 'p<0.001'
        ax.text(0.05, 0.95, f'R²={r2:.2f}, {p_str}',
                transform=ax.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

        if row_idx == 2: ax.set_xlabel(pl, fontsize=10)
        else:            ax.set_xlabel('')
        if col_idx == 0: ax.set_ylabel(hl, fontsize=9)
        else:            ax.set_ylabel('')
        if row_idx == 0: ax.set_title(pl, fontsize=11, fontweight='bold')

fig.suptitle('Health Indicators vs PISA 2022 Scores (n=43 countries)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('facet_grid.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Multivariate OLS Regression

We regress each PISA subject score on all three health indicators simultaneously. This allows us to assess the **partial association** of each indicator while holding the others constant, and to evaluate **joint explanatory power** (R²).

Predictors are **standardised** (z-scores) so coefficients are directly comparable across indicators.

In [ ]:
# Standardise all variables (z-scores)
df_z = df.copy()
for col in pisa_vars + health_vars:
    df_z[col] = (df[col] - df[col].mean()) / df[col].std()

X = sm.add_constant(df_z[health_vars])
X.columns = ['Intercept', 'Life Expectancy', 'Infant Mortality', 'Health Expenditure']

results = {}
for pv in pisa_vars:
    results[pv] = sm.OLS(df_z[pv], X).fit()

# Summary table
rows = []
for pv, res in results.items():
    subject = pv.replace('_score', '').title()
    for coef_name in ['Life Expectancy', 'Infant Mortality', 'Health Expenditure']:
        rows.append({
            'PISA Subject':     subject,
            'Predictor':        coef_name,
            'β (std.)':         round(res.params[coef_name], 3),
            'Std. Error':       round(res.bse[coef_name], 3),
            't-stat':           round(res.tvalues[coef_name], 2),
            'p-value':          round(res.pvalues[coef_name], 3),
            'Sig.':             star(res.pvalues[coef_name]),
            'R²':               round(res.rsquared, 3),
            'Adj. R²':          round(res.rsquared_adj, 3),
        })

reg_table = pd.DataFrame(rows)
print("Multivariate OLS Regression — standardised coefficients")
print("Dependent variable: standardised PISA score\n")
reg_table

In [ ]:
# Coefficient plot
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
colors = {'Life Expectancy': 'steelblue', 'Infant Mortality': 'tomato', 'Health Expenditure': 'seagreen'}

for ax, pv in zip(axes, pisa_vars):
    res = results[pv]
    predictors = ['Life Expectancy', 'Infant Mortality', 'Health Expenditure']
    coefs  = [res.params[p] for p in predictors]
    errors = [res.bse[p] * 1.96 for p in predictors]  # 95% CI

    y_pos = range(len(predictors))
    ax.barh(y_pos, coefs, xerr=errors, align='center',
            color=[colors[p] for p in predictors], alpha=0.8,
            edgecolor='white', height=0.5)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(predictors, fontsize=9)
    ax.set_title(f"{pv.replace('_score','').title()}\n(R²={res.rsquared:.2f})",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Standardised coefficient (β)', fontsize=9)

fig.suptitle('OLS Regression Coefficients with 95% CI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('regression_coefficients.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Robustness Checks

### 5a. Log-transformation of infant mortality

Infant mortality is right-skewed (range: 1.4–17.5). A log transformation linearises the relationship with PISA scores and reduces the leverage of extreme values.

In [ ]:
df['log_infant_mortality'] = np.log(df['infant_mortality'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
                           ['infant_mortality', 'log_infant_mortality'],
                           ['Raw infant mortality', 'Log infant mortality']):
    ax.hist(df[col], bins=14, edgecolor='white', color='steelblue', alpha=0.85)
    ax.set_title(title, fontsize=11)

plt.suptitle('Effect of log-transformation on infant mortality distribution',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Compare Pearson r — raw vs log
rows = []
for pv in pisa_vars:
    r_raw,  p_raw  = stats.pearsonr(df['infant_mortality'],     df[pv])
    r_log,  p_log  = stats.pearsonr(df['log_infant_mortality'], df[pv])
    rows.append({
        'PISA subject':             pv.replace('_score','').title(),
        'r (raw infant mort.)':     f'{r_raw:.3f} (p={p_raw:.3f})',
        'r (log infant mort.)':     f'{r_log:.3f} (p={p_log:.3f})',
    })
print("Pearson r: raw vs log-transformed infant mortality")
pd.DataFrame(rows)

### 5b. Sensitivity analysis — excluding the United States

The United States is a well-documented outlier: it has the highest health expenditure among OECD countries (~16.5% of GDP) but PISA scores and life expectancy below the OECD average (Deaton, 2013). We check whether the results are sensitive to its inclusion.

In [ ]:
df_no_us = df[df['country'] != 'United States'].reset_index(drop=True)

rows = []
for hv in health_vars:
    for pv in pisa_vars:
        r_full, p_full = stats.pearsonr(df[hv],       df[pv])
        r_nous, p_nous = stats.pearsonr(df_no_us[hv], df_no_us[pv])
        rows.append({
            'Health indicator': hv.replace('_', ' ').title(),
            'PISA subject':     pv.replace('_score', '').title(),
            f'r  full (n={len(df)})':    f'{r_full:.3f}{star(p_full)}',
            f'r  excl. US (n={len(df_no_us)})': f'{r_nous:.3f}{star(p_nous)}',
            'Direction stable': '✓' if np.sign(r_full) == np.sign(r_nous) else '✗',
        })

print("Sensitivity: full sample vs excluding the United States")
print("* p<0.05  ** p<0.01  *** p<0.001")
pd.DataFrame(rows)

In [ ]:
# Visual: health expenditure vs math — with and without US
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, data, title in zip(axes,
                            [df, df_no_us],
                            [f'Full sample (n={len(df)})',
                             f'Excluding United States (n={len(df_no_us)})']):
    r, p = stats.pearsonr(data['health_exp_pct_gdp'], data['math_score'])
    sns.regplot(data=data, x='health_exp_pct_gdp', y='math_score', ax=ax,
                scatter_kws=dict(s=50, alpha=0.75, edgecolors='white'),
                line_kws=dict(color='firebrick', linewidth=1.5), ci=95)
    for _, row in data[data['country'].isin(['United States','Japan','Korea','Finland','Mexico'])].iterrows():
        ax.annotate(row['country'], xy=(row['health_exp_pct_gdp'], row['math_score']),
                    xytext=(4, 2), textcoords='offset points', fontsize=8)
    p_str = f'p={p:.3f}' if p >= 0.001 else 'p<0.001'
    ax.text(0.05, 0.95, f'R²={r**2:.2f}, {p_str}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Health Expenditure (% of GDP)', fontsize=10)
    ax.set_ylabel('Math Score (PISA points)', fontsize=10)

fig.suptitle('Sensitivity: Health Expenditure vs Math Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sensitivity_us.png', bbox_inches='tight', dpi=150)
plt.show()

## Summary of Findings

| Finding | Result |
|---------|--------|
| Infant mortality ↔ PISA | Strong **negative** association (r ≈ −0.80); robust across all subjects and to log-transformation |
| Life expectancy ↔ PISA | Moderate **positive** association (r ≈ +0.63); consistent across subjects |
| Health expenditure ↔ PISA | Weak **positive** in full sample; **weakens further** after removing the US outlier |
| Multivariate R² | The three health indicators jointly explain **~70%** of variance in PISA scores |
| US outlier | US has the highest health expenditure (~16.5% GDP) but below-average PISA scores and life expectancy — consistent with Deaton (2013) |
| Robustness | Spearman and Pearson correlations align closely; log-transformation of infant mortality does not change substantive conclusions |